In [2]:
# ============================================================
# 00 — CONNECT GOOGLE DRIVE / PROJECT
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
# ============================================================
# PROJECT PATH
# ============================================================

from pathlib import Path
import sys

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Exists: {PROJECT_DIR.exists()}")

Project directory: /content/drive/MyDrive/FitnessML_Master
Exists: True


In [10]:
# ============================================================
# ADD PROJECT TO PYTHON PATH
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project added to Python path.")

Project added to Python path.


In [14]:
# ============================================================
# 01 — FITBIT DATASET AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import config_fitbit as config

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

print("config_fitbit loaded successfully")
print(f"Raw data directory: {config.RAW_DATA_DIR}")

print("=" * 70)
print("FITBIT DATASET AUDIT")
print("=" * 70)

print(f"Data directory: {config.RAW_DATA_DIR}")

config_fitbit loaded successfully
Raw data directory: /content/drive/MyDrive/FitnessML_Master/data/raw/fitbit
FITBIT DATASET AUDIT
Data directory: /content/drive/MyDrive/FitnessML_Master/data/raw/fitbit


In [15]:
# ============================================================
# DISCOVER RAW FILES
# ============================================================

RAW_DIR = Path(config.RAW_DATA_DIR)

csv_files = sorted(RAW_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")
print()

for path in csv_files:
    print(f"{path.name:45s} {path.stat().st_size / 1024**2:8.2f} MB")

CSV files found: 18

dailyActivity_merged.csv                          0.11 MB
dailyCalories_merged.csv                          0.02 MB
dailyIntensities_merged.csv                       0.07 MB
dailySteps_merged.csv                             0.02 MB
heartrate_seconds_merged.csv                     85.44 MB
hourlyCalories_merged.csv                         0.76 MB
hourlyIntensities_merged.csv                      0.86 MB
hourlySteps_merged.csv                            0.76 MB
minuteCaloriesNarrow_merged.csv                  63.37 MB
minuteCaloriesWide_merged.csv                    21.93 MB
minuteIntensitiesNarrow_merged.csv               44.21 MB
minuteIntensitiesWide_merged.csv                  3.16 MB
minuteMETsNarrow_merged.csv                      45.48 MB
minuteSleep_merged.csv                            8.44 MB
minuteStepsNarrow_merged.csv                     44.38 MB
minuteStepsWide_merged.csv                        3.32 MB
sleepDay_merged.csv                               0

In [16]:
# ============================================================
# COLUMN DETECTION
# ============================================================

def find_column(columns, candidates):
    """
    Find first matching column from a list of candidates.
    Matching is case-insensitive.
    """

    normalized = {
        str(col).strip().lower(): col
        for col in columns
    }

    for candidate in candidates:
        candidate_norm = candidate.strip().lower()

        if candidate_norm in normalized:
            return normalized[candidate_norm]

    return None


def detect_structure(df):
    """
    Detect user/date/time columns without assuming
    identical schemas across Fitbit CSV files.
    """

    user_col = find_column(
        df.columns,
        ["Id", "user_id", "UserId", "userId"]
    )

    date_col = find_column(
        df.columns,
        [
            "Date",
            "ActivityDate",
            "SleepDay",
            "date",
            "datetime",
            "DateTime"
        ]
    )

    time_col = find_column(
        df.columns,
        [
            "Time",
            "time",
            "timestamp",
            "Timestamp"
        ]
    )

    return user_col, date_col, time_col

In [17]:
# ============================================================
# SINGLE FILE AUDIT
# ============================================================

def audit_file(path, chunksize=200_000):

    result = {
        "file": path.name,
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "rows": 0,
        "columns": 0,
        "users": np.nan,
        "date_min": pd.NaT,
        "date_max": pd.NaT,
        "duplicates": np.nan,
        "missing_cells": np.nan,
        "user_column": None,
        "date_column": None,
        "time_column": None,
    }

    # --------------------------------------------------------
    # Read header first
    # --------------------------------------------------------

    header = pd.read_csv(path, nrows=0)

    result["columns"] = len(header.columns)

    user_col, date_col, time_col = detect_structure(header)

    result["user_column"] = user_col
    result["date_column"] = date_col
    result["time_column"] = time_col

    # --------------------------------------------------------
    # Process file in chunks
    # --------------------------------------------------------

    unique_users = set()
    min_date = None
    max_date = None

    duplicate_count = 0
    missing_cells = 0

    for chunk in pd.read_csv(
        path,
        chunksize=chunksize,
        low_memory=False
    ):

        result["rows"] += len(chunk)

        missing_cells += int(chunk.isna().sum().sum())

        if user_col and user_col in chunk.columns:
            unique_users.update(
                chunk[user_col].dropna().unique()
            )

        if date_col and date_col in chunk.columns:

            dates = pd.to_datetime(
                chunk[date_col],
                errors="coerce"
            )

            if dates.notna().any():

                chunk_min = dates.min()
                chunk_max = dates.max()

                if min_date is None or chunk_min < min_date:
                    min_date = chunk_min

                if max_date is None or chunk_max > max_date:
                    max_date = chunk_max

        # Exact duplicate rows inside each chunk
        duplicate_count += int(chunk.duplicated().sum())

    result["users"] = len(unique_users) if user_col else np.nan
    result["date_min"] = min_date
    result["date_max"] = max_date
    result["duplicates"] = duplicate_count
    result["missing_cells"] = missing_cells

    return result

In [18]:
# ============================================================
# AUDIT ALL FILES
# ============================================================

audit_results = []

for i, path in enumerate(csv_files, start=1):

    print(
        f"[{i:02d}/{len(csv_files)}] "
        f"Auditing {path.name} ..."
    )

    try:
        result = audit_file(path)
        audit_results.append(result)

    except Exception as e:
        print(f"   ERROR: {e}")

        audit_results.append({
            "file": path.name,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "rows": np.nan,
            "columns": np.nan,
            "users": np.nan,
            "date_min": pd.NaT,
            "date_max": pd.NaT,
            "duplicates": np.nan,
            "missing_cells": np.nan,
            "user_column": None,
            "date_column": None,
            "time_column": None,
        })

audit_df = pd.DataFrame(audit_results)

print()
print("=" * 70)
print("FILE AUDIT COMPLETE")
print("=" * 70)

display(audit_df)

[01/18] Auditing dailyActivity_merged.csv ...
[02/18] Auditing dailyCalories_merged.csv ...
[03/18] Auditing dailyIntensities_merged.csv ...
[04/18] Auditing dailySteps_merged.csv ...
[05/18] Auditing heartrate_seconds_merged.csv ...
[06/18] Auditing hourlyCalories_merged.csv ...
[07/18] Auditing hourlyIntensities_merged.csv ...
[08/18] Auditing hourlySteps_merged.csv ...
[09/18] Auditing minuteCaloriesNarrow_merged.csv ...
[10/18] Auditing minuteCaloriesWide_merged.csv ...
[11/18] Auditing minuteIntensitiesNarrow_merged.csv ...
[12/18] Auditing minuteIntensitiesWide_merged.csv ...
[13/18] Auditing minuteMETsNarrow_merged.csv ...
[14/18] Auditing minuteSleep_merged.csv ...
[15/18] Auditing minuteStepsNarrow_merged.csv ...
[16/18] Auditing minuteStepsWide_merged.csv ...
[17/18] Auditing sleepDay_merged.csv ...


/tmp/ipykernel_466/1079290415.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(


[18/18] Auditing weightLogInfo_merged.csv ...

FILE AUDIT COMPLETE


/tmp/ipykernel_466/1079290415.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(


,file,size_mb,rows,columns,users,date_min,date_max,duplicates,missing_cells,user_column,date_column,time_column
0,dailyActivity_merged.csv,0.11,940,15,33,2016-04-12 00:00:00,2016-05-12 00:00:00,0,0,Id,ActivityDate,None
1,dailyCalories_merged.csv,0.02,940,3,33,NaT,NaT,0,0,Id,None,None
2,dailyIntensities_merged.csv,0.07,940,10,33,NaT,NaT,0,0,Id,None,None
3,dailySteps_merged.csv,0.02,940,3,33,NaT,NaT,0,0,Id,None,None
4,heartrate_seconds_merged.csv,85.44,2483658,3,14,NaT,NaT,0,0,Id,None,Time
5,hourlyCalories_merged.csv,0.76,22099,3,33,NaT,NaT,0,0,Id,None,None
6,hourlyIntensities_merged.csv,0.86,22099,4,33,NaT,NaT,0,0,Id,None,None
7,hourlySteps_merged.csv,0.76,22099,3,33,NaT,NaT,0,0,Id,None,None
8,minuteCaloriesNarrow_merged.csv,63.37,1325580,3,33,NaT,NaT,0,0,Id,None,None
9,minuteCaloriesWide_merged.csv,21.93,21645,62,33,NaT,NaT,0,0,Id,None,None


In [19]:
# ============================================================
# USER COVERAGE ACROSS FILES
# ============================================================

file_users = {}

for path in csv_files:

    try:
        header = pd.read_csv(path, nrows=0)
        user_col, _, _ = detect_structure(header)

        if user_col is None:
            continue

        users = set()

        for chunk in pd.read_csv(
            path,
            usecols=[user_col],
            chunksize=200_000
        ):
            users.update(
                chunk[user_col].dropna().unique()
            )

        file_users[path.name] = users

    except Exception as e:
        print(f"{path.name}: {e}")

In [20]:
# ============================================================
# USER COVERAGE SUMMARY
# ============================================================

coverage = []

for filename, users in file_users.items():

    coverage.append({
        "file": filename,
        "users": len(users)
    })

coverage_df = (
    pd.DataFrame(coverage)
    .sort_values("users", ascending=False)
)

print("=" * 70)
print("USER COVERAGE")
print("=" * 70)

display(coverage_df)

USER COVERAGE


,file,users
0,dailyActivity_merged.csv,33
1,dailyCalories_merged.csv,33
2,dailyIntensities_merged.csv,33
3,dailySteps_merged.csv,33
5,hourlyCalories_merged.csv,33
6,hourlyIntensities_merged.csv,33
8,minuteCaloriesNarrow_merged.csv,33
7,hourlySteps_merged.csv,33
9,minuteCaloriesWide_merged.csv,33
10,minuteIntensitiesNarrow_merged.csv,33


In [21]:
# ============================================================
# USER OVERLAP
# ============================================================

core_files = [
    "dailyActivity_merged.csv",
    "heartRate_seconds_merged.csv",
    "sleepDay_merged.csv",
    "weightLogInfo_merged.csv"
]

available_core = [
    name for name in core_files
    if name in file_users
]

print("=" * 70)
print("USER OVERLAP — CORE FILES")
print("=" * 70)

for i in range(len(available_core)):

    for j in range(i + 1, len(available_core)):

        a = available_core[i]
        b = available_core[j]

        overlap = file_users[a] & file_users[b]

        print(
            f"{a:35s} × {b:35s} "
            f"→ {len(overlap)} common users"
        )

USER OVERLAP — CORE FILES
dailyActivity_merged.csv            × sleepDay_merged.csv                 → 24 common users
dailyActivity_merged.csv            × weightLogInfo_merged.csv            → 8 common users
sleepDay_merged.csv                 × weightLogInfo_merged.csv            → 6 common users


In [23]:
# ============================================================
# LOAD CORE TABLES
# ============================================================

daily_activity = pd.read_csv(
    RAW_DIR / "dailyActivity_merged.csv"
)

heart_rate = pd.read_csv(
    RAW_DIR / "heartrate_seconds_merged.csv"
)

sleep_day = pd.read_csv(
    RAW_DIR / "sleepDay_merged.csv"
)

weight_log = pd.read_csv(
    RAW_DIR / "weightLogInfo_merged.csv"
)

print("=" * 70)
print("CORE TABLES LOADED")
print("=" * 70)

print(f"daily_activity : {daily_activity.shape}")
print(f"heart_rate     : {heart_rate.shape}")
print(f"sleep_day      : {sleep_day.shape}")
print(f"weight_log     : {weight_log.shape}")

CORE TABLES LOADED
daily_activity : (940, 15)
heart_rate     : (2483658, 3)
sleep_day      : (413, 5)
weight_log     : (67, 8)


In [24]:
# ============================================================
# IMPORTANT:
# NO CROSS-TABLE MERGE IS PERFORMED IN THIS NOTEBOOK.
#
# Different Fitbit tables have different temporal granularities
# and different user/date coverage.
#
# A unified longitudinal dataset will be constructed only after
# the audit determines the appropriate temporal resolution,
# target variable and usable user/date intersection.
# ============================================================

print("=" * 70)
print("DATASET AUDIT COMPLETED")
print("=" * 70)

print("""
No cross-table merge was performed.

Next step:
1. Evaluate data quality and temporal coverage.
2. Determine the usable user/date intersection.
3. Select temporal granularity.
4. Define the target variable.
5. Construct the modeling dataset.
""")

DATASET AUDIT COMPLETED

No cross-table merge was performed.

Next step:
1. Evaluate data quality and temporal coverage.
2. Determine the usable user/date intersection.
3. Select temporal granularity.
4. Define the target variable.
5. Construct the modeling dataset.



FILE: dailyActivity
------------------------------------------------
columns
dtypes
missing %
unique users
date column
date range
duplicate rows

In [26]:
# ============================================================
# CORE FILE SCHEMA
# ============================================================

core_paths = {
    "daily_activity": RAW_DIR / "dailyActivity_merged.csv",
    "heart_rate": RAW_DIR / "heartrate_seconds_merged.csv",
    "sleep_day": RAW_DIR / "sleepDay_merged.csv",
    "weight_log": RAW_DIR / "weightLogInfo_merged.csv",
}

for name, path in core_paths.items():

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    df_sample = pd.read_csv(path, nrows=5)

    print("\nColumns:")
    for col in df_sample.columns:
        print(f"  {col}")

    print("\nDtypes:")
    print(df_sample.dtypes)


DAILY_ACTIVITY

Columns:
  Id
  ActivityDate
  TotalSteps
  TotalDistance
  TrackerDistance
  LoggedActivitiesDistance
  VeryActiveDistance
  ModeratelyActiveDistance
  LightActiveDistance
  SedentaryActiveDistance
  VeryActiveMinutes
  FairlyActiveMinutes
  LightlyActiveMinutes
  SedentaryMinutes
  Calories

Dtypes:
Id                            int64
ActivityDate                 object
TotalSteps                    int64
TotalDistance               float64
TrackerDistance             float64
LoggedActivitiesDistance      int64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance       int64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
dtype: object

HEART_RATE

Columns:
  Id
  Time
  Value

Dtypes:
Id        int64
Time     object
Value     int64
dtype: object

SLEEP_DAY

C

In [28]:
# ============================================================
# MISSING VALUES — CORE FILES
# ============================================================

for name, path in core_paths.items():

    print("\n" + "=" * 70)
    print(f"MISSING VALUES: {name.upper()}")
    print("=" * 70)

    df = pd.read_csv(path)

    missing = pd.DataFrame({
        "missing": df.isna().sum(),
        "missing_%": df.isna().mean() * 100
    })

    display(
        missing[missing["missing"] > 0]
        .sort_values("missing", ascending=False)
    )


MISSING VALUES: DAILY_ACTIVITY


,missing,missing_%



MISSING VALUES: HEART_RATE


,missing,missing_%



MISSING VALUES: SLEEP_DAY


,missing,missing_%



MISSING VALUES: WEIGHT_LOG


,missing,missing_%
Fat,65,97.014925


In [29]:
# ============================================================
# WEIGHT LOG INSPECTION
# ============================================================

print("=" * 70)
print("WEIGHT LOG SAMPLE")
print("=" * 70)

display(
    weight_log[
        [
            "Id",
            "Date",
            "WeightKg",
            "WeightPounds",
            "BMI",
            "IsManualReport"
        ]
    ].sort_values(["Id", "Date"])
)

WEIGHT LOG SAMPLE


,Id,Date,WeightKg,WeightPounds,BMI,IsManualReport
0,1503960366,5/2/2016 11:59:59 PM,52.599998,115.963147,22.650000,True
1,1503960366,5/3/2016 11:59:59 PM,52.599998,115.963147,22.650000,True
2,1927972279,4/13/2016 1:08:52 AM,133.500000,294.317120,47.540001,False
3,2873212765,4/21/2016 11:59:59 PM,56.700001,125.002104,21.450001,True
4,2873212765,5/12/2016 11:59:59 PM,57.299999,126.324875,21.690001,True
5,4319703577,4/17/2016 11:59:59 PM,72.400002,159.614681,27.450001,True
6,4319703577,5/4/2016 11:59:59 PM,72.300003,159.394222,27.379999,True
7,4558609924,4/18/2016 11:59:59 PM,69.699997,153.662190,27.250000,True
8,4558609924,4/25/2016 11:59:59 PM,70.300003,154.984977,27.459999,True
9,4558609924,5/1/2016 11:59:59 PM,69.900002,154.103125,27.320000,True


In [30]:
# ============================================================
# WEIGHT RECORDS PER USER
# ============================================================

weight_records = (
    weight_log
    .groupby("Id")
    .agg(
        records=("Id", "size"),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        weight_min=("WeightKg", "min"),
        weight_max=("WeightKg", "max"),
        weight_mean=("WeightKg", "mean"),
    )
    .reset_index()
)

display(weight_records)

,Id,records,first_date,last_date,weight_min,weight_max,weight_mean
0,1503960366,2,5/2/2016 11:59:59 PM,5/3/2016 11:59:59 PM,52.599998,52.599998,52.599998
1,1927972279,1,4/13/2016 1:08:52 AM,4/13/2016 1:08:52 AM,133.500000,133.500000,133.500000
2,2873212765,2,4/21/2016 11:59:59 PM,5/12/2016 11:59:59 PM,56.700001,57.299999,57.000000
3,4319703577,2,4/17/2016 11:59:59 PM,5/4/2016 11:59:59 PM,72.300003,72.400002,72.350002
4,4558609924,5,4/18/2016 11:59:59 PM,5/9/2016 11:59:59 PM,69.099998,70.300003,69.639999
5,5577150313,1,4/17/2016 9:17:55 AM,4/17/2016 9:17:55 AM,90.699997,90.699997,90.699997
6,6962181067,30,4/12/2016 11:59:59 PM,5/9/2016 11:59:59 PM,61.000000,62.500000,61.553334
7,8877689391,24,4/12/2016 6:47:11 AM,5/9/2016 6:39:44 AM,84.000000,85.800003,85.145834


In [31]:
# ============================================================
# TEMPORAL COVERAGE — DAILY ACTIVITY
# ============================================================

daily_activity["ActivityDate"] = pd.to_datetime(
    daily_activity["ActivityDate"]
)

user_temporal = (
    daily_activity
    .groupby("Id")
    .agg(
        records=("Id", "size"),
        first_date=("ActivityDate", "min"),
        last_date=("ActivityDate", "max"),
        unique_days=("ActivityDate", "nunique"),
    )
    .reset_index()
)

user_temporal["calendar_span"] = (
    user_temporal["last_date"]
    - user_temporal["first_date"]
).dt.days + 1

user_temporal["coverage_pct"] = (
    user_temporal["unique_days"]
    / user_temporal["calendar_span"]
    * 100
)

display(
    user_temporal
    .sort_values("unique_days", ascending=False)
)

,Id,records,first_date,last_date,unique_days,calendar_span,coverage_pct
0,1503960366,31,2016-04-12,2016-05-12,31,31,100.0
1,1624580081,31,2016-04-12,2016-05-12,31,31,100.0
3,1844505072,31,2016-04-12,2016-05-12,31,31,100.0
4,1927972279,31,2016-04-12,2016-05-12,31,31,100.0
5,2022484408,31,2016-04-12,2016-05-12,31,31,100.0
7,2320127002,31,2016-04-12,2016-05-12,31,31,100.0
6,2026352035,31,2016-04-12,2016-05-12,31,31,100.0
12,4020332650,31,2016-04-12,2016-05-12,31,31,100.0
9,2873212765,31,2016-04-12,2016-05-12,31,31,100.0
16,4445114986,31,2016-04-12,2016-05-12,31,31,100.0


In [32]:
# ============================================================
# MISSING DAYS INSIDE USER TIMELINES
# ============================================================

user_temporal["missing_days"] = (
    user_temporal["calendar_span"]
    - user_temporal["unique_days"]
)

display(
    user_temporal[
        [
            "Id",
            "unique_days",
            "calendar_span",
            "missing_days",
            "coverage_pct"
        ]
    ]
    .sort_values("coverage_pct")
)

,Id,unique_days,calendar_span,missing_days,coverage_pct
0,1503960366,31,31,0,100.0
1,1624580081,31,31,0,100.0
2,1644430081,30,30,0,100.0
3,1844505072,31,31,0,100.0
4,1927972279,31,31,0,100.0
5,2022484408,31,31,0,100.0
6,2026352035,31,31,0,100.0
7,2320127002,31,31,0,100.0
8,2347167796,18,18,0,100.0
9,2873212765,31,31,0,100.0
